In [1]:
import os
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Adding AWS S3 Minio configs
sparkConf = (
    SparkConf()
    .set("spark.jars.ivy","/home/brijeshdhaker/.ivy2")
    .set("spark.ui.port", "4040")
    #.set("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.0.0,io.delta:delta-spark_2.12:3.3.2")
    #.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    #.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    #.set("spark.executor.heartbeatInterval", "300000")
    #.set("spark.network.timeout", "400000")
    #.set("spark.hadoop.fs.s3a.endpoint", "http://minio.sandbox.net:9010")
    #.set("spark.hadoop.fs.s3a.access.key", "pgm2H2bR7a5kMc5XCYdO")
    #.set("spark.hadoop.fs.s3a.secret.key", "zjd8T0hXFGtfemVQ6AH3yBAPASJNXNbVSx5iddqG")
    #.set("spark.hadoop.fs.s3a.path.style.access", "true")
    #.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    #.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    #.set("spark.eventLog.enabled", "true")
    #.set("spark.eventLog.dir", "file:///apps/var/logs/spark-events")
)

spark = (
    SparkSession.builder.master("local[*]").
        appName('spark-sql-notebook').
        config(conf=sparkConf).
        getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/30 09:56:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:


# sample data for dataframe
sampleData = (("Ram", "Sales", 3000),
              ("Meena", "Sales", 4600),
              ("Robin", "Sales", 4100),
              ("Kunal", "Finance", 3000),
              ("Vinod", "Sales", 3000),
              ("Srishti", "Management", 3300),
              ("Jeny", "Finance", 3900),
              ("Hitesh", "Marketing", 3000),
              ("Kailash", "Marketing", 2000),
              ("Sharad", "Sales", 4100)
              )

# column names for dataframe
columns = ["Employee_Name", "Department", "Salary"]

dataFrame = spark.createDataFrame(data=sampleData, schema=columns)
dataFrame.show()

# importing Window from pyspark.sql.window
from pyspark.sql.window import Window

# creating a window partition of dataframe
windowPartitionAgg  = Window.partitionBy("Department")

""" Aggregate Functions """
dataFrame.withColumn("Avg", avg(col("salary")).over(windowPartitionAgg)) \
    .withColumn("Sum", sum(col("salary")).over(windowPartitionAgg)) \
    .withColumn("Min", min(col("salary")).over(windowPartitionAgg)) \
    .withColumn("Max", max(col("salary")).over(windowPartitionAgg)) \
    .show()

+-------------+----------+------+
|Employee_Name|Department|Salary|
+-------------+----------+------+
|          Ram|     Sales|  3000|
|        Meena|     Sales|  4600|
|        Robin|     Sales|  4100|
|        Kunal|   Finance|  3000|
|          Ram|     Sales|  3000|
|      Srishti|Management|  3300|
|         Jeny|   Finance|  3900|
|       Hitesh| Marketing|  3000|
|      Kailash| Marketing|  2000|
|       Sharad|     Sales|  4100|
+-------------+----------+------+

+-------------+----------+------+------+-----+----+----+
|Employee_Name|Department|Salary|   Avg|  Sum| Min| Max|
+-------------+----------+------+------+-----+----+----+
|        Kunal|   Finance|  3000|3450.0| 6900|3000|3900|
|         Jeny|   Finance|  3900|3450.0| 6900|3000|3900|
|      Srishti|Management|  3300|3300.0| 3300|3300|3300|
|       Hitesh| Marketing|  3000|2500.0| 5000|2000|3000|
|      Kailash| Marketing|  2000|2500.0| 5000|2000|3000|
|          Ram|     Sales|  3000|3760.0|18800|3000|4600|
|        M

#### Window Ranking Functions

In [ ]:
# sample data for dataframe
sampleData = ((101, "Ram", "Biology", 80),
              (103, "Meena", "Social Science", 78),
              (104, "Robin", "Sanskrit", 58),
              (102, "Kunal", "Physics", 89),
              (101, "Vinod", "Biology", 80),
              (106, "Srishti", "Maths", 70),
              (108, "Jeny", "Physics", 75),
              (107, "Hitesh", "Maths", 88),
              (109, "Kailash", "Maths", 90),
              (105, "Sharad", "Social Science", 84)
              )

# column names for dataframe
columns = ["Roll_No", "Student_Name", "Subject", "Marks"]

df = spark.createDataFrame(data=sampleData, schema=columns)
df.show()

# importing Window from pyspark.sql.window
from pyspark.sql.window import Window
windowPartition = Window.partitionBy("Subject").orderBy("Marks")


+-------+------------+--------------+-----+
|Roll_No|Student_Name|       Subject|Marks|
+-------+------------+--------------+-----+
|    101|         Ram|       Biology|   80|
|    103|       Meena|Social Science|   78|
|    104|       Robin|      Sanskrit|   58|
|    102|       Kunal|       Physics|   89|
|    101|       Vinod|       Biology|   80|
|    106|     Srishti|         Maths|   70|
|    108|        Jeny|       Physics|   75|
|    107|      Hitesh|         Maths|   88|
|    109|     Kailash|         Maths|   90|
|    105|      Sharad|Social Science|   84|
+-------+------------+--------------+-----+



##### row_number


In [21]:
# importing row_number() from pyspark.sql.functions
from pyspark.sql.functions import row_number

# applying the row_number() function
df.withColumn("row_number",
               row_number().over(windowPartition)).show()

+-------+------------+--------------+-----+----------+
|Roll_No|Student_Name|       Subject|Marks|row_number|
+-------+------------+--------------+-----+----------+
|    101|         Ram|       Biology|   80|         1|
|    101|       Vinod|       Biology|   80|         2|
|    106|     Srishti|         Maths|   70|         1|
|    107|      Hitesh|         Maths|   88|         2|
|    109|     Kailash|         Maths|   90|         3|
|    108|        Jeny|       Physics|   75|         1|
|    102|       Kunal|       Physics|   89|         2|
|    104|       Robin|      Sanskrit|   58|         1|
|    103|       Meena|Social Science|   78|         1|
|    105|      Sharad|Social Science|   84|         2|
+-------+------------+--------------+-----+----------+




##### rank


In [16]:
# importing rank() from pyspark.sql.functions
from pyspark.sql.functions import rank

# applying the rank() function
df.withColumn("rank", rank().over(windowPartition)) \
    .show()

+-------+------------+--------------+-----+----+
|Roll_No|Student_Name|       Subject|Marks|rank|
+-------+------------+--------------+-----+----+
|    101|         Ram|       Biology|   80|   1|
|    101|       Vinod|       Biology|   80|   1|
|    106|     Srishti|         Maths|   70|   1|
|    107|      Hitesh|         Maths|   88|   2|
|    109|     Kailash|         Maths|   90|   3|
|    108|        Jeny|       Physics|   75|   1|
|    102|       Kunal|       Physics|   89|   2|
|    104|       Robin|      Sanskrit|   58|   1|
|    103|       Meena|Social Science|   78|   1|
|    105|      Sharad|Social Science|   84|   2|
+-------+------------+--------------+-----+----+



##### percent_rank


In [10]:
# importing percent_rank() from pyspark.sql.functions
from pyspark.sql.functions import percent_rank

# applying the percent_rank() function
df.withColumn("percent_rank", percent_rank().over(windowPartition)) \
    .show()

+-------------+---+----------+------+------------+
|Employee_Name|Age|Department|Salary|percent_rank|
+-------------+---+----------+------+------------+
|        Kunal| 25|   Finance|  3000|         0.0|
|         Jeny| 26|   Finance|  3900|         1.0|
|      Srishti| 46|Management|  3300|         0.0|
|      Kailash| 29| Marketing|  2000|         0.0|
|       Hitesh| 30| Marketing|  3000|         1.0|
|          Ram| 28|     Sales|  3000|         0.0|
|          Ram| 28|     Sales|  3000|         0.0|
|        Meena| 33|     Sales|  4600|         0.5|
|       Sharad| 39|     Sales|  4100|        0.75|
|        Robin| 40|     Sales|  4100|         1.0|
+-------------+---+----------+------+------------+



##### dense_rank

In [11]:
# importing dense_rank() from pyspark.sql.functions
from pyspark.sql.functions import dense_rank

# applying the dense_rank() function
df.withColumn("dense_rank", dense_rank().over(windowPartition)) \
    .show()

+-------------+---+----------+------+----------+
|Employee_Name|Age|Department|Salary|dense_rank|
+-------------+---+----------+------+----------+
|        Kunal| 25|   Finance|  3000|         1|
|         Jeny| 26|   Finance|  3900|         2|
|      Srishti| 46|Management|  3300|         1|
|      Kailash| 29| Marketing|  2000|         1|
|       Hitesh| 30| Marketing|  3000|         2|
|          Ram| 28|     Sales|  3000|         1|
|          Ram| 28|     Sales|  3000|         1|
|        Meena| 33|     Sales|  4600|         2|
|       Sharad| 39|     Sales|  4100|         3|
|        Robin| 40|     Sales|  4100|         4|
+-------------+---+----------+------+----------+



### Window Analytics Functions

In [3]:
# Create sample data for dataframe
sampleData = (("Ram", 28, "Sales", 3000),
              ("Meena", 33, "Sales", 4600),
              ("Robin", 40, "Sales", 4100),
              ("Kunal", 25, "Finance", 3000),
              ("Ram", 28, "Sales", 3000),
              ("Srishti", 46, "Management", 3300),
              ("Jeny", 26, "Finance", 3900),
              ("Hitesh", 30, "Marketing", 3000),
              ("Kailash", 29, "Marketing", 2000),
              ("Sharad", 39, "Sales", 4100)
              )

# column names for dataframe
columns = ["Employee_Name", "Age", "Department", "Salary"]

df = spark.createDataFrame(data=sampleData, schema=columns)
df.show()

# importing Window from pyspark.sql.window
from pyspark.sql.window import Window
windowPartition = Window.partitionBy("Department").orderBy("Age")



+-------------+---+----------+------+
|Employee_Name|Age|Department|Salary|
+-------------+---+----------+------+
|          Ram| 28|     Sales|  3000|
|        Meena| 33|     Sales|  4600|
|        Robin| 40|     Sales|  4100|
|        Kunal| 25|   Finance|  3000|
|          Ram| 28|     Sales|  3000|
|      Srishti| 46|Management|  3300|
|         Jeny| 26|   Finance|  3900|
|       Hitesh| 30| Marketing|  3000|
|      Kailash| 29| Marketing|  2000|
|       Sharad| 39|     Sales|  4100|
+-------------+---+----------+------+



##### cume_dist 

In [4]:
# importing cume_dist()
# from pyspark.sql.functions
from pyspark.sql.functions import cume_dist

# applying window function with the help of DataFrame.withColumn
df.withColumn("cume_dist", cume_dist().over(windowPartition)).show()

+-------------+---+----------+------+---------+
|Employee_Name|Age|Department|Salary|cume_dist|
+-------------+---+----------+------+---------+
|        Kunal| 25|   Finance|  3000|      0.5|
|         Jeny| 26|   Finance|  3900|      1.0|
|      Srishti| 46|Management|  3300|      1.0|
|      Kailash| 29| Marketing|  2000|      0.5|
|       Hitesh| 30| Marketing|  3000|      1.0|
|          Ram| 28|     Sales|  3000|      0.4|
|          Ram| 28|     Sales|  3000|      0.4|
|        Meena| 33|     Sales|  4600|      0.6|
|       Sharad| 39|     Sales|  4100|      0.8|
|        Robin| 40|     Sales|  4100|      1.0|
+-------------+---+----------+------+---------+



##### lag

In [5]:
# importing lag() from pyspark.sql.functions
from pyspark.sql.functions import lag

df.withColumn("Lag", lag("Salary", 2).over(windowPartition)) \
    .show()

+-------------+---+----------+------+----+
|Employee_Name|Age|Department|Salary| Lag|
+-------------+---+----------+------+----+
|        Kunal| 25|   Finance|  3000|NULL|
|         Jeny| 26|   Finance|  3900|NULL|
|      Srishti| 46|Management|  3300|NULL|
|      Kailash| 29| Marketing|  2000|NULL|
|       Hitesh| 30| Marketing|  3000|NULL|
|          Ram| 28|     Sales|  3000|NULL|
|          Ram| 28|     Sales|  3000|NULL|
|        Meena| 33|     Sales|  4600|3000|
|       Sharad| 39|     Sales|  4100|3000|
|        Robin| 40|     Sales|  4100|4600|
+-------------+---+----------+------+----+



##### lead


In [6]:
# importing lead() from pyspark.sql.functions
from pyspark.sql.functions import lead

df.withColumn("Lead", lead("salary", 2).over(windowPartition)) \
    .show()

+-------------+---+----------+------+----+
|Employee_Name|Age|Department|Salary|Lead|
+-------------+---+----------+------+----+
|        Kunal| 25|   Finance|  3000|NULL|
|         Jeny| 26|   Finance|  3900|NULL|
|      Srishti| 46|Management|  3300|NULL|
|      Kailash| 29| Marketing|  2000|NULL|
|       Hitesh| 30| Marketing|  3000|NULL|
|          Ram| 28|     Sales|  3000|4600|
|          Ram| 28|     Sales|  3000|4100|
|        Meena| 33|     Sales|  4600|4100|
|       Sharad| 39|     Sales|  4100|NULL|
|        Robin| 40|     Sales|  4100|NULL|
+-------------+---+----------+------+----+

